## Attention Mechanism

This allows that every 'word' in the sentences gets to 'give/modify' the meaning of the other words in the sentence by the corresponding meaning, this is done on the embedding space.

video: https://www.youtube.com/watch?v=wjZofJX0v4M


Matrices to accomplish this: 


Query: 

Key:

Value:



Steps: 

- Compute the attention scores 
- Compute the attention weights (nosmalize version of the scores)
- Comput the context vector (How the contex modifies the current token)

### Attention with NO PARAMETERS (No Learning)

In [ ]:
import torch

# Input example
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)


query = inputs[1]
# Attention -> context -> modifications
attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
  attn_scores_2[i] = torch.dot(x_i, query)

print("Scores ")
print(attn_scores_2)
print("\n")
attn_weights = torch.softmax(attn_scores_2, dim=0)

print("Weights ")
print(attn_weights)
print("\n")

#  Apply the modifications based on the context (on each dimension dimension)
context_vec = torch.zeros(query.shape)
for i,x_i in enumerate(inputs):
    context_vec += attn_weights[i]*x_i

print("Context - Modifications ")
print(context_vec)
print("\n")

Scores 
tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


Weights 
tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])


Context - Modifications 
tensor([0.4419, 0.6515, 0.5683])




### Using matrix multiplication (more efficient)

In [11]:
att_scores = (torch.softmax(inputs @ inputs.T, dim=1)) @ inputs
print(att_scores)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


## Attention Mechanism with weights

In [30]:
d_in = inputs.shape[1]  # the input embedding size, d=3
d_out = 2               # the output embedding size, d=2 (MApping of the query space)


torch.manual_seed(123)

W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key   = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

query = inputs @ W_query
key = inputs @ W_key 
value = inputs @ W_value

att_score_3 = torch.softmax(((query @ key.T)/d_out**0.5), dim=1) @ value


print(att_score_3)

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]])


### Implementing a self attention class

In [39]:
import torch.nn as nn

class selfAttention(nn.Module):
  def __init__(self, d_in, d_out):
      super().__init__()
      self.W_query = nn.Parameter(torch.rand(d_in, d_out))
      self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
      self.W_value = nn.Parameter(torch.rand(d_in, d_out))

  def forward(self, x):
    keys = x @ self.W_key
    queries = x @ self.W_query
    values = x @ self.W_value
    
    attn_scores = queries @ keys.T # omega
    attn_weights = torch.softmax(
        attn_scores / keys.shape[-1]**0.5, dim=-1
    )

    context_vec = attn_weights @ values
    return context_vec

In [40]:
torch.manual_seed(123)
sa_v1 = selfAttention(d_in, d_out)
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


In [42]:
class SelfAttention_v2(nn.Module):

    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        # Better weight initialization that Parameters, more stable training
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


### Hiiding future words 

we want the predictions only to depend on the past words.. also the meaning of one word at the momento of predicting the next one just to deppend on the meaning of the past once. for this masking the future words is usefull, this way the model doesn't take them into account at the attention and prediction.

In [48]:
class CausalAttention(nn.Module):
  def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
    super().__init__()
    self.d_out = dropout
    self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.dropout = nn.Dropout(dropout) # Drop out layer to put on top of the attention block 
    self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

  def forward(self, x):
    b, num_tokens, d_in = x.shape

    keys = self.W_key(x)
    queries = self.W_query(x)
    values = self.W_value(x)

    # Attention scores 
    att_scores = queries @ keys.transpose(1,2)

    # Dropout mask
    # `:num_tokens` to account for cases where the number of tokens in the batch is smaller than the supported context_size
    att_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)

    # Attention weights with dropout
    att_weights = self.dropout(torch.softmax(att_scores / keys.shape[-1]**0.5, dim=-1))

    # Context matrix
    context = att_weights @ values
    return context
  

#Test
torch.manual_seed(123)
batch = torch.stack((inputs, inputs), dim=0)  # 2 batches of, 6 tokens, 3 embedding dim
print(inputs.shape)   # 6 tokens, 3 embedding dim
print(batch.shape)    # 2 batches of, 6 tokens, 3 embedding dim
context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0.0)

context_vecs = ca(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)



torch.Size([6, 3])
torch.Size([2, 6, 3])
tensor([[[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]],

        [[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]]], grad_fn=<UnsafeViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


## Multi-Head attention

Add multiple heads of attention to extract more 'relations/knowledge' from the texts and then concatenate the result.

For efficiency the W_query, W_key, and W_value are creating for all heads and then each head uses the part of the matrix that corresponds to them

In [75]:
class MultiHeadAttention(nn.Module):
  def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
    super().__init__()
    assert (d_out % num_heads == 0), "d_out must be divisible by num_heads"

    self.d_out = d_out
    self.num_heads = num_heads
    self.head_dim = d_out // num_heads
    self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
    self.dropout = nn.Dropout(dropout)
    self.register_buffer(
        "mask",
        torch.triu(torch.ones(context_length, context_length),
                    diagonal=1)
    )


  def forward(self, x):
    b, num_tokens, _ = x.shape

    keys = self.W_key(x)          # Shape: (b, num_tokens, d_out)
    queries = self.W_query(x)
    values = self.W_value(x)

    # Split the matrix on the heads 
    keys = keys.view(b, num_tokens, self.num_heads, self.head_dim) 
    values = values.view(b, num_tokens, self.num_heads, self.head_dim)
    queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

    # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
    keys = keys.transpose(1, 2)
    queries = queries.transpose(1, 2)
    values = values.transpose(1, 2)

    # Attention scores 
    attn_scores = queries @ keys.transpose(2, 3)

    # Mask 
    mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
    attn_scores.masked_fill_(mask_bool, -torch.inf)

    # Attention weights
    attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
    attn_weights = self.dropout(attn_weights)

    # Context
    context = (attn_weights @ values).transpose(1, 2) 

    # Combine all the heads 
    context = context.contiguous().view(b, num_tokens, self.d_out)
    context = self.out_proj(context)

    return context





In [76]:
torch.manual_seed(123)

print(batch.shape)
batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

torch.Size([2, 6, 3])
tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])
